# LODES Performance Benchmark: DuckDB Parquet vs PostGIS

Compares two query paths for LODES8 employment data:

| Path | Function | Storage |
|------|----------|---------|
| PostGIS | `lodes.tract_employment()` | PostgreSQL table |
| DuckDB | `duckdb_analytics.employment_by_sector()` | Local parquet cache |

**Setup**: Collin County TX (`state_fips="48"`, `county_fips="085"`, `year=2021`).
Run `geostack load-lodes --state 48 --county 085 --year 2021` before executing.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from geostack.db import get_engine
from geostack.duckdb_analytics import commute_distance_distribution, employment_by_sector
from geostack.lodes import commute_shed, tract_employment

warnings.filterwarnings("ignore")

STATE_FIPS = "48"
COUNTY_FIPS = "085"
YEAR = 2021
N_RUNS = 3

engine = get_engine()
print("Engine ready")

## Benchmark helpers

In [ ]:
def benchmark(fn, n=N_RUNS, label=""):
    """Run fn n times, return dict with min/mean/max elapsed seconds."""
    times = []
    for i in range(n):
        t0 = time.perf_counter()
        result = fn()
        times.append(time.perf_counter() - t0)
    stats = {"label": label, "min": min(times), "mean": np.mean(times), "max": max(times), "runs": n}
    print(f"{label:50s}  min={stats['min']:.3f}s  mean={stats['mean']:.3f}s  max={stats['max']:.3f}s")
    return stats, result

results = {}

## Benchmark 1: Employment by Sector

**PostGIS path** — `lodes.tract_employment()` joins the `lodes_wac` table against `census_tracts`, returning tract-level aggregates.

**DuckDB path** — `duckdb_analytics.employment_by_sector()` reads the local `.parquet` cache column-selectively with no DB round-trip.

In [ ]:
# PostGIS — tract_employment
stats_pg_emp, df_pg_emp = benchmark(
    lambda: tract_employment(STATE_FIPS, COUNTY_FIPS, YEAR, engine=engine),
    label="PostGIS  tract_employment()",
)
results["pg_employment"] = stats_pg_emp

# DuckDB — employment_by_sector
stats_db_emp, df_db_emp = benchmark(
    lambda: employment_by_sector(STATE_FIPS, YEAR, f"{STATE_FIPS}{COUNTY_FIPS}"),
    label="DuckDB   employment_by_sector()",
)
results["db_employment"] = stats_db_emp

print(f"\nPostGIS rows: {len(df_pg_emp)}  |  DuckDB rows: {len(df_db_emp)}")

## Benchmark 2: Commute Shed

**PostGIS path** — `lodes.commute_shed()` queries the `lodes_od` table, aggregating OD flows by home county.

**DuckDB path** — `duckdb_analytics.commute_distance_distribution()` scans the OD parquet file directly.

In [ ]:
# PostGIS — commute_shed
stats_pg_od, df_pg_od = benchmark(
    lambda: commute_shed(STATE_FIPS, COUNTY_FIPS, YEAR, engine=engine),
    label="PostGIS  commute_shed()",
)
results["pg_commute"] = stats_pg_od

# DuckDB — commute_distance_distribution
stats_db_od, df_db_od = benchmark(
    lambda: commute_distance_distribution(STATE_FIPS, YEAR, f"{STATE_FIPS}{COUNTY_FIPS}"),
    label="DuckDB   commute_distance_distribution()",
)
results["db_commute"] = stats_db_od

print(f"\nPostGIS rows: {len(df_pg_od)}  |  DuckDB rows: {len(df_db_od)}")

## Results Table

In [ ]:
summary = pd.DataFrame(results).T[["label", "min", "mean", "max"]]
summary[["min", "mean", "max"]] = summary[["min", "mean", "max"]].round(3)
summary.index = ["PostGIS employment", "DuckDB employment", "PostGIS commute", "DuckDB commute"]
summary

## Bar Chart: Mean Query Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, (pg_key, db_key, title) in zip(
    axes,
    [
        ("pg_employment", "db_employment", "Employment by Sector"),
        ("pg_commute", "db_commute", "Commute Shed"),
    ],
):
    pg_times = results[pg_key]
    db_times = results[db_key]

    bars = ax.bar(
        ["PostGIS", "DuckDB"],
        [pg_times["mean"], db_times["mean"]],
        color=["#4477AA", "#EE6677"],
        width=0.5,
        yerr=[[pg_times["mean"] - pg_times["min"], db_times["mean"] - db_times["min"]],
              [pg_times["max"] - pg_times["mean"], db_times["max"] - db_times["mean"]]],
        capsize=5,
        error_kw={"elinewidth": 1.5},
    )
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{bar.get_height():.3f}s",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel("Mean elapsed (s)")
    ax.set_ylim(0)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    f"LODES Query Performance — Collin County TX ({YEAR}, {N_RUNS} runs)",
    fontsize=12,
    y=1.02,
)
plt.tight_layout()
plt.savefig("benchmark_lodes.png", dpi=150, bbox_inches="tight")
plt.show()

## When to Use Each Path

### DuckDB (parquet cache)
- **Best for**: Exploratory analysis, large county-wide or state-wide scans, iterative ad-hoc queries during development.
- **Why fast**: Reads only the columns it needs, uses vectorized SIMD operations, no network round-trip or connection overhead.
- **Limitations**: No geometry — you get tabular data only. Requires the parquet cache files to be present locally.
  Use `geostack load-lodes` to populate the cache.

### PostGIS
- **Best for**: Spatial queries that join employment data to geometry (e.g., "jobs within 0.5 miles of a site"),
  production pipelines where data must be centrally authoritative, and queries that combine LODES with
  parcels/roads/ACS data that lives only in the database.
- **Why sometimes slower**: Network round-trip, query planner overhead, row-based storage for wide tables.
- **Advantage**: GIST spatial indexes make `ST_Within`, `ST_Intersects`, and buffer queries extremely fast —
  a capability DuckDB parquet cannot match without geometry columns.

**Rule of thumb**: Use DuckDB first for exploration; switch to PostGIS when you need geometry joins or
when queries will run in production alongside other spatial data.